<a href="https://colab.research.google.com/github/Moehassan09/DS3001_NFL_Predictor/blob/main/DS3001_Group_Project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Requisite Libraries

In [ ]:
# Install modern, compatible versions that play nice with Python 3.11
!pip install "numpy<2.0.0" "pandas>=2.0.0" nfl_data_py xgboost matplotlib seaborn --quiet

# Restart the runtime automatically
import os
os.kill(os.getpid(), 9)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 1.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.3/18.3 MB 48.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 24.9 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
opencv-python-headless 4.12.0.88 requires numpy<2.3.0,>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
thinc 8.3.6 requires numpy<3.0.0,>=2.0.0, but you have numpy 1.26.4 which is incompatible.


In [ ]:
import numpy as np
import pandas as pd
import nfl_data_py as nfl

print(f"NumPy version: {np.__version__}") # Should be 1.26.x
print(f"Pandas version: {pd.__version__}")

# Test the data connection
test_df = nfl.import_schedules([2023])
print("Connection Successful!")

Data Acquisition

In [ ]:
import nfl_data_py as nfl
import pandas as pd

# 1. Download Schedules (Contains the Spread/Betting lines)
# We go back to 1999 as per project requirements
years = list(range(1999, 2026))
schedules = nfl.import_schedules(years)

# 2. Download Play-by-Play (To get EPA - Expected Points Added)
# Optimization: Only pull the columns we need to prevent memory crashes
pbp_cols = ['game_id', 'home_team', 'away_team', 'epa', 'home_wp', 'away_wp']
pbp = nfl.import_pbp_data(years, columns=pbp_cols)

# 3. Aggregate EPA by Game
# We need to know the total offensive efficiency for the home and away teams for every game
home_epa = pbp.groupby(['game_id', 'home_team'])['epa'].sum().reset_index().rename(columns={'epa': 'home_epa', 'home_team': 'team'})
away_epa = pbp.groupby(['game_id', 'away_team'])['epa'].sum().reset_index().rename(columns={'epa': 'away_epa', 'away_team': 'team'})

# 4. Merge everything into the master dataframe 'df'
df = schedules.merge(home_epa[['game_id', 'home_epa']], on='game_id', how='inner')
df = df.merge(away_epa[['game_id', 'away_epa']], on='game_id', how='inner')

# 5. Final Cleaning: Remove games that haven't happened yet (NaN scores)
df = df.dropna(subset=['home_score', 'away_score', 'spread_line'])

print(f"Data Load Complete. Total games prepared: {len(df)}")

Code

In [ ]:
import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.metrics import brier_score_loss

def run_vectorized_nfl_pipeline(df):
    """
    High-performance pipeline using vectorized operations for sports prediction.
    Target: 1 if Home Team Covers, 0 otherwise.
    """
    # --- 1. Vectorized Target Creation ---
    # result = home_score - away_score.
    # spread_line is home handicap (e.g., -7 means home is 7pt favorite).
    df['home_cover'] = ((df['result'] + df['spread_line']) > 0).astype(int)

    # --- 2. Long-Form Transformation for EPA ---
    # To calculate rolling stats efficiently, we 'melt' the data so each game
    # has two rows: one for the home team and one for the away team.
    home_cols = ['game_id', 'season', 'week', 'home_team', 'home_epa']
    away_cols = ['game_id', 'season', 'week', 'away_team', 'away_epa']

    home_df = df[home_cols].rename(columns={'home_team': 'team', 'home_epa': 'epa'})
    away_df = df[away_cols].rename(columns={'away_team': 'team', 'away_epa': 'epa'})

    long_df = pd.concat([home_df, away_df]).sort_values(['team', 'season', 'week'])

    # --- 3. Vectorized Rolling EPA (Zero Leakage) ---
    # .groupby('team')['epa'] isolates each team's performance history.
    # .shift(1) ensures the model only sees games COMPLETELY FINISHED before kickoff.
    # .rolling(5).mean() captures the 'Form' of the team.
    long_df['rolling_epa'] = (
        long_df.groupby('team')['epa']
        .transform(lambda x: x.shift(1).rolling(5).mean())
    )

    # --- 4. Broadcasting Stats back to Schedule ---
    # We merge the team-specific rolling stats back onto the main game rows.
    df = df.merge(
        long_df[['game_id', 'team', 'rolling_epa']],
        left_on=['game_id', 'home_team'], right_on=['game_id', 'team']
    ).rename(columns={'rolling_epa': 'home_rolling_epa'}).drop('team', axis=1)

    df = df.merge(
        long_df[['game_id', 'team', 'rolling_epa']],
        left_on=['game_id', 'away_team'], right_on=['game_id', 'team']
    ).rename(columns={'rolling_epa': 'away_rolling_epa'}).drop('team', axis=1)

    # --- 5. Clean & Split ---
    df_ml = df.dropna(subset=['home_rolling_epa', 'away_rolling_epa'])
    train = df_ml[df_ml['season'] < 2024]
    test = df_ml[df_ml['season'] == 2024]

    # --- 6. Modeling ---
    features = ['spread_line', 'home_rolling_epa', 'away_rolling_epa']
    model = xgb.XGBClassifier(n_estimators=100, learning_rate=0.03, max_depth=3)
    model.fit(train[features], train['home_cover'])

    # Probabilistic evaluation
    test['prob'] = model.predict_proba(test[features])[:, 1]
    brier = brier_score_loss(test['home_cover'], test['prob'])

    print(f"Vectorized Pipeline Complete. Brier Score: {brier:.4f}")
    return model, test
run_vectorized_nfl_pipeline(df)

EDA Visualization

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

def plot_spread_efficiency(df):
    """
    Visualizes the relationship between the Sportsbook Spread
    and the actual Margin of Victory.
    """
    plt.figure(figsize=(12, 8))

    # Set a professional style
    sns.set_theme(style="whitegrid")

    # Create the regression plot
    # x = spread_line (negative if home is favorite)
    # y = result (home_score - away_score)
    plot = sns.regplot(
        data=df.sample(2000), # Sample 2000 games for visual clarity
        x='spread_line',
        y='result',
        scatter_kws={'alpha':0.3, 'color':'teal'},
        line_kws={'color':'red', 'label':'Market Trend'}
    )

    # Add context lines for 'The Push' (where margin exactly equals spread)
    plt.axhline(0, color='black', linestyle='--', alpha=0.5)
    plt.axvline(0, color='black', linestyle='--', alpha=0.5)

    plt.title("NFL Market Efficiency: Closing Spread vs. Actual Margin (1999-Present)", fontsize=16)
    plt.xlabel("Closing Spread (Home Perspective)", fontsize=12)
    plt.ylabel("Actual Margin of Victory (Home Score - Away Score)", fontsize=12)
    plt.legend()

    plt.show()

# Run the visualization
plot_spread_efficiency(df)